# ScenePatch — Gemma 4 E2B reproducibility notebook

This notebook is the primary executable Gemma path for ScenePatch: a controlled synthetic before/after fixture pack with generated spoken-intent audio, the official `google/gemma-4-E2B-it` checkpoint, three native function declarations, and a deterministic validator that never dynamically executes a model-selected function. Public-use approval for the exact five media hashes is recorded in `docs/FIXTURE_RIGHTS.md`.

**Why the notebook is primary:** on 31 July 2026, the pinned browser q4f16 runtime loaded and emitted native calls on the release M4 Pro, but it proposed a commit for the controlled missing-blue-marker fixture. That fails ScenePatch's release gate. The public Pages UI must therefore remain a clearly labeled scripted fixture replay unless a later tagged browser build passes the complete gate.

**No benchmark result is embedded here.** The notebook records only values produced by the current execution. The public-use approval in `docs/FIXTURE_RIGHTS.md` applies only while the five media hashes remain unchanged.

### Kaggle setup

1. Attach Kaggle's official Google Gemma 4 Transformers `gemma-4-e2b-it` V1 model input and enable a GPU accelerator. Internet is used only to install the Python dependencies, with Transformers pinned to 5.14.1; the attached model weights load from Kaggle input storage with `local_files_only=True`.
2. Attach the controlled-synthetic dataset named `scenepatch-controlled-fixture` containing `scene-before.png`, `scene-after-bad.png`, `scene-after-corrected.png`, `scene-after-occluded.png`, and `intent.wav`. The exact five media hashes are approved for the repository, Kaggle dataset/notebook, and demo video; regenerated or edited media require new approval.
3. Run all cells. The model loads once, then the notebook preprocesses and records the bad, corrected, and occluded cases separately. The full checkpoint is large, so this is a transparent reproducibility path rather than a browser-latency claim.

References: [Gemma 4 E2B IT model card](https://huggingface.co/google/gemma-4-E2B-it) and [Gemma 4 function calling](https://ai.google.dev/gemma/docs/capabilities/text/function-calling-gemma4).

In [ ]:
%pip install -q -U "transformers==5.14.1" accelerate librosa soundfile pillow jsonschema

In [ ]:
from __future__ import annotations

from collections.abc import Mapping
from pathlib import Path
import hashlib
import json
import platform
import re
import time
import unicodedata

import kagglehub
import librosa
import soundfile as sf
import torch
import transformers
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import Audio, display
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "google/gemma-4-E2B-it"
MODEL_REVISION = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"
KAGGLE_MODEL_HANDLE = "google/gemma-4/transformers/gemma-4-e2b-it/1"
MODEL_IDENTIFIER = f"kaggle://{KAGGLE_MODEL_HANDLE}"
UPSTREAM_REFERENCE = f"{MODEL_ID}@{MODEL_REVISION} (documented reference; not byte-verified against Kaggle V1)"
MAX_AUDIO_SECONDS = 10.0
SAMPLE_RATE = 16_000

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before loading the full checkpoint.")
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model_id": MODEL_IDENTIFIER,
    "upstream_reference": UPSTREAM_REFERENCE,
})

## Native tool declarations and fail-closed validator

Gemma proposes calls; the code below only validates data. It does not use `eval`, `globals()`, imports, shell commands, network calls, or any function name supplied by the model. A parse or validator failure receives exactly one constrained correction attempt; a valid semantic decision is never retried, and a second malformed result remains blocked. A clean proposal remains pending until a human confirms it in the application.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "record_change",
            "description": "Record one visible scene change relative to the spoken intent.",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {
                        "type": "string",
                        "description": "Short, observable description of the change.",
                    },
                    "classification": {
                        "type": "string",
                        "enum": ["intended", "unexplained", "uncertain"],
                        "description": "How this change relates to the spoken intent.",
                    },
                },
                "required": ["description", "classification"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "commit_patch",
            "description": "Propose a patch only when every visible change is intended.",
            "parameters": {
                "type": "object",
                "properties": {"summary": {"type": "string"}},
                "required": ["summary"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "block_commit",
            "description": "Propose a block when any change is unexplained or uncertain.",
            "parameters": {
                "type": "object",
                "properties": {"reason": {"type": "string"}},
                "required": ["reason"],
                "additionalProperties": False,
            },
        },
    },
]

ALLOWED_NAMES = {"record_change", "commit_patch", "block_commit"}
CLASSIFICATIONS = {"intended", "unexplained", "uncertain"}


def _object_arguments(value):
    if isinstance(value, Mapping):
        return dict(value)
    if isinstance(value, str):
        decoded = json.loads(value)
        if isinstance(decoded, Mapping):
            return dict(decoded)
    raise ValueError("Tool arguments must be a JSON object.")


def normalize_tool_calls(parsed_response):
    if not isinstance(parsed_response, Mapping):
        raise ValueError("Parsed response must be an object.")
    raw_calls = parsed_response.get("tool_calls")
    if not isinstance(raw_calls, list):
        raise ValueError("Response must contain a tool_calls list.")

    normalized = []
    for item in raw_calls:
        if not isinstance(item, Mapping):
            raise ValueError("Each tool call must be an object.")
        payload = item.get("function", item)
        if not isinstance(payload, Mapping):
            raise ValueError("Each function payload must be an object.")
        name = payload.get("name")
        if name not in ALLOWED_NAMES:
            raise ValueError(f"Unknown tool: {name!r}")
        normalized.append({"name": name, "arguments": _object_arguments(payload.get("arguments"))})
    return normalized


def _bounded_text(value, field, maximum):
    if not isinstance(value, str):
        raise ValueError(f"{field} must be a string.")
    cleaned = " ".join(value.split())
    if not 1 <= len(cleaned) <= maximum:
        raise ValueError(f"{field} must contain 1 to {maximum} characters.")
    return cleaned


def validate_and_decide(parsed_response):
    calls = normalize_tool_calls(parsed_response)
    if len(calls) > 7:
        raise ValueError("At most six changes and one terminal call are allowed.")

    changes, terminals, seen = [], [], set()
    for call in calls:
        name, arguments = call["name"], call["arguments"]
        if name == "record_change":
            if set(arguments) != {"description", "classification"}:
                raise ValueError("record_change has missing or extra arguments.")
            description = _bounded_text(arguments["description"], "description", 280)
            classification = arguments["classification"]
            if classification not in CLASSIFICATIONS:
                raise ValueError(f"Invalid classification: {classification!r}")
            duplicate_key = re.sub(r"[.!?]+$", "", unicodedata.normalize("NFKC", description)).casefold()
            if duplicate_key in seen:
                raise ValueError("Duplicate change description.")
            seen.add(duplicate_key)
            changes.append({"description": description, "classification": classification})
        else:
            expected = {"summary"} if name == "commit_patch" else {"reason"}
            if set(arguments) != expected:
                raise ValueError(f"{name} has missing or extra arguments.")
            field = next(iter(expected))
            terminals.append({"name": name, field: _bounded_text(arguments[field], field, 500)})

    if not 1 <= len(changes) <= 6:
        raise ValueError("One to six change records are required.")
    if len(terminals) != 1:
        raise ValueError("Exactly one terminal call is required.")

    unsafe = [c for c in changes if c["classification"] in {"unexplained", "uncertain"}]
    terminal = terminals[0]
    if unsafe:
        decision = "blocked_by_deterministic_policy"
    elif terminal["name"] == "block_commit":
        decision = "blocked_as_proposed"
    else:
        decision = "pending_human_confirmation"

    return {"valid": True, "changes": changes, "terminal": terminal, "decision": decision}


In [ ]:
# Deterministic policy checks; these are code tests, not model-quality results.
clean_sample = {"tool_calls": [
    {"function": {"name": "record_change", "arguments": {"description": "Red marker moved above the sketchbook", "classification": "intended"}}},
    {"function": {"name": "commit_patch", "arguments": {"summary": "Requested marker move only"}}},
]}
unsafe_sample = {"tool_calls": [
    {"function": {"name": "record_change", "arguments": {"description": "Blue marker is missing", "classification": "unexplained"}}},
    {"function": {"name": "commit_patch", "arguments": {"summary": "Model proposed commit"}}},
]}
if validate_and_decide(clean_sample)["decision"] != "pending_human_confirmation":
    raise AssertionError("A clean validated proposal must remain pending human confirmation.")
if validate_and_decide(unsafe_sample)["decision"] != "blocked_by_deterministic_policy":
    raise AssertionError("The deterministic policy must override an unsafe commit proposal.")
try:
    validate_and_decide({"tool_calls": [{"function": {"name": "delete_file", "arguments": {}}}]})
except ValueError as error:
    if "Unknown tool" not in str(error):
        raise
else:
    raise AssertionError("Unknown tools must fail closed.")
print("Deterministic validator checks passed.")

## Load and normalize the controlled synthetic fixture

The next cell intentionally stops if the exact approved generated fixture pack is absent. It does not download or substitute other media, and it verifies the five media hashes before inference.

In [ ]:
FIXTURE_ROOT = Path("/kaggle/input/scenepatch-controlled-fixture")
BEFORE_PATH = FIXTURE_ROOT / "scene-before.png"
AFTER_CASES = {
    "bad": FIXTURE_ROOT / "scene-after-bad.png",
    "corrected": FIXTURE_ROOT / "scene-after-corrected.png",
    "occluded": FIXTURE_ROOT / "scene-after-occluded.png",
}
AUDIO_PATH = FIXTURE_ROOT / "intent.wav"

required_files = [BEFORE_PATH, *AFTER_CASES.values(), AUDIO_PATH]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Attach the controlled synthetic scenepatch-controlled-fixture dataset. Missing: " + ", ".join(missing)
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_hashes = {path.name: sha256_file(path) for path in required_files}
if len(input_hashes) != len(required_files):
    raise ValueError("Fixture filenames must be unique.")
print(json.dumps(input_hashes, indent=2))
APPROVED_HASHES = {
    "scene-before.png": "4822982cf8d254fa3b4579ab40c72131627c92a2780f77da7490ad865b7d83cf",
    "scene-after-bad.png": "71210ba4d515b1abe4a1c3e134e22fd9d9b2c7f1f1ea7b1396e96b155e2e2120",
    "scene-after-corrected.png": "9a22ca251072aee8d79d84ba4f4562f46400a5fec651bc3dae19ba4929e60d03",
    "scene-after-occluded.png": "b16ff8c9d41b65522fc0d72c8a9b108f72353c54315950fa4460c23e6370b5d3",
    "intent.wav": "d99fe8ce17149e9c8bc94e90467a95d3e1f5b98ba67e7fbc7f2c8fe9f201b020",
}
if input_hashes != APPROVED_HASHES:
    raise ValueError("Fixture hashes do not match the approved release record.")
print("All five media hashes match the approved public-use record.")

In [ ]:
WORK_ROOT = Path("/kaggle/working/scenepatch")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
NORMALIZED_AUDIO = WORK_ROOT / "intent-16khz-mono.wav"

def render_panel(source_path, label):
    with Image.open(source_path) as opened:
        source = ImageOps.exif_transpose(opened).convert("RGB")
    scale = max(512 / source.width, 512 / source.height)
    resized = source.resize((round(source.width * scale), round(source.height * scale)), Image.Resampling.LANCZOS)
    panel = ImageOps.fit(resized, (512, 512), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    draw = ImageDraw.Draw(panel)
    draw.rectangle((0, 0, 511, 57), fill="#101412")
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf", 22)
    draw.text((24, 18), label, fill="#d8ff63" if label.startswith("BEFORE") else "#8ad9ff", font=font)
    return panel


def build_contact_sheet(before_path, after_path, output_path):
    sheet = Image.new("RGB", (1024, 512), "white")
    sheet.paste(render_panel(before_path, "BEFORE · BASE"), (0, 0))
    sheet.paste(render_panel(after_path, "AFTER · WORKTREE"), (512, 0))
    ImageDraw.Draw(sheet).line((512, 0, 512, 512), fill="#777777", width=2)
    if sheet.size != (1024, 512):
        raise ValueError("Contact sheet must be exactly 1024 by 512 pixels.")
    sheet.save(output_path, format="WEBP", quality=90, method=6)
    return {
        "path": str(output_path),
        "width": sheet.width,
        "height": sheet.height,
        "sha256": sha256_file(output_path),
    }


source_audio_info = sf.info(AUDIO_PATH)
if not 0 < source_audio_info.duration <= MAX_AUDIO_SECONDS:
    raise ValueError(f"The spoken intent must be non-empty and at most {MAX_AUDIO_SECONDS:.0f} seconds.")
waveform, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
if waveform.size == 0:
    raise ValueError("Intent audio is empty.")
sf.write(NORMALIZED_AUDIO, waveform, SAMPLE_RATE, subtype="PCM_16")
audio_record = {
    "source_filename": AUDIO_PATH.name,
    "source_seconds": source_audio_info.duration,
    "processed_seconds": len(waveform) / SAMPLE_RATE,
    "sample_rate_hz": SAMPLE_RATE,
    "channels": 1,
    "source_sha256": input_hashes[AUDIO_PATH.name],
    "processed_sha256": sha256_file(NORMALIZED_AUDIO),
}

case_preprocessing = {}
for case_name, after_path in AFTER_CASES.items():
    contact_sheet_path = WORK_ROOT / f"before-after-{case_name}.webp"
    case_preprocessing[case_name] = {
        "before": {"filename": BEFORE_PATH.name, "sha256": input_hashes[BEFORE_PATH.name]},
        "after": {"filename": after_path.name, "sha256": input_hashes[after_path.name]},
        "contact_sheet": build_contact_sheet(BEFORE_PATH, after_path, contact_sheet_path),
        "audio": audio_record,
    }
    print(f"Prepared case: {case_name}")
    print(json.dumps(case_preprocessing[case_name], indent=2))
    display(Image.open(contact_sheet_path))

display(Audio(filename=str(NORMALIZED_AUDIO)))

## Load the official checkpoint

This is Kaggle's V1 copy of the unquantized official Google checkpoint and is separate from the browser's ONNX runtime. The attached model input resolves through Kaggle's local resource cache; no token is embedded and model loading is fail-closed with `local_files_only=True`.

In [ ]:
load_started = time.perf_counter()
MODEL_ROOT = Path(kagglehub.model_download(KAGGLE_MODEL_HANDLE))
for required_name in ("config.json", "processor_config.json", "tokenizer.json", "model.safetensors"):
    if not (MODEL_ROOT / required_name).is_file():
        raise FileNotFoundError(f"Attached Gemma model is missing {required_name}: {MODEL_ROOT}")
processor = AutoProcessor.from_pretrained(str(MODEL_ROOT), local_files_only=True)
model = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_ROOT),
    local_files_only=True,
    dtype="auto",
    device_map="auto",
)
model.eval()
model_load_seconds = time.perf_counter() - load_started
print({"model_id": MODEL_IDENTIFIER, "upstream_reference": UPSTREAM_REFERENCE, "model_root": str(MODEL_ROOT), "load_seconds_this_run": model_load_seconds, "device": str(model.device)})

In [ ]:
SYSTEM_PROMPT = """You are ScenePatch's visual change reviewer for a small creative desk.
Compare the labeled BEFORE and AFTER panels with the spoken intent. Silently inventory every object in BEFORE, verify that it remains present in AFTER, then check additions and location changes. Record each material visible change once, and never record unchanged objects.
Classify a requested change as intended, an unrequested change as unexplained, and ambiguous evidence as uncertain.
After recording changes, call exactly one terminal tool. Commit only if every change is intended; otherwise block.
Do not make safety, identity, theft, inventory, or forensic claims."""

CORRECTION_INSTRUCTION = (
    "Correction: return only native calls to record_change followed by exactly one "
    "commit_patch or block_commit. Use one to six unique changes, only the declared "
    "classifications, valid arguments, and no prose or extra fields."
)


def prepare_case_inputs(contact_sheet_path, previous_raw_output=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "path": str(contact_sheet_path)},
                {"type": "text", "text": "Review this scene change against the spoken intent and use only the declared tools."},
                {"type": "audio", "audio": str(NORMALIZED_AUDIO)},
            ],
        },
    ]
    if previous_raw_output is not None:
        messages.extend([
            {"role": "assistant", "content": previous_raw_output},
            {"role": "user", "content": CORRECTION_INSTRUCTION},
        ])
    return processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)

In [ ]:
def blocked_audit(stage, error):
    return {
        "valid": False,
        "changes": [],
        "terminal": None,
        "decision": "blocked_invalid_output",
        "failure": {"stage": stage, **error},
    }


def parse_and_audit(raw_output, prefix):
    try:
        parsed_output = processor.parse_response(raw_output, prefix=prefix)
    except Exception as error:
        failure = {"stage": "parse", "type": type(error).__name__, "message": str(error)}
        return None, failure, blocked_audit("parse", {"type": failure["type"], "message": failure["message"]})

    try:
        return parsed_output, None, validate_and_decide(parsed_output)
    except Exception as error:
        failure = {"stage": "validate", "type": type(error).__name__, "message": str(error)}
        return parsed_output, failure, blocked_audit("validate", {"type": failure["type"], "message": failure["message"]})


case_generations = {}
for case_name, preprocessing in case_preprocessing.items():
    attempts = []
    previous_raw_output = None
    for attempt_number in (1, 2):
        inputs = prepare_case_inputs(
            preprocessing["contact_sheet"]["path"],
            previous_raw_output=previous_raw_output,
        )
        input_length = inputs["input_ids"].shape[-1]

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        inference_started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        inference_seconds = time.perf_counter() - inference_started

        generated_tokens = generated[0][input_length:]
        raw_output = processor.decode(generated_tokens, skip_special_tokens=False)
        parsed_output, failure, audit = parse_and_audit(raw_output, inputs["input_ids"])
        attempt_record = {
            "attempt": attempt_number,
            "kind": "initial" if attempt_number == 1 else "constrained_correction",
            "correction_instruction": CORRECTION_INSTRUCTION if attempt_number == 2 else None,
            "input_keys": sorted(inputs.keys()),
            "input_tokens": int(input_length),
            "generated_tokens": int(generated_tokens.shape[-1]),
            "inference_seconds_this_run": inference_seconds,
            "raw_output": raw_output,
            "parsed_output": parsed_output,
            "failure": failure,
            "audit": audit,
        }
        attempts.append(attempt_record)

        print(f"\n=== CASE: {case_name} · ATTEMPT {attempt_number} ===")
        print("Raw Gemma generation:")
        print(raw_output)
        print("\nParsed response and audit:")
        print(json.dumps({"parsed_output": parsed_output, "failure": failure, "audit": audit}, indent=2, default=str))
        print({"inference_seconds_this_run": inference_seconds, "generated_tokens": int(generated_tokens.shape[-1])})
        del inputs, generated, generated_tokens

        if audit["valid"]:
            break
        previous_raw_output = raw_output

    selected_attempt = attempts[-1]
    case_generations[case_name] = {
        "attempts": attempts,
        "retry_used": len(attempts) == 2,
        "selected_attempt": selected_attempt["attempt"],
        "total_inference_seconds_this_run": sum(
            attempt["inference_seconds_this_run"] for attempt in attempts
        ),
    }

In [ ]:
# The last attempt is selected. A retry exists only when attempt one was malformed or failed validation.
execution_records = {}
for case_name, preprocessing in case_preprocessing.items():
    generation = case_generations[case_name]
    attempts = generation["attempts"]
    selected = attempts[-1]
    if len(attempts) == 2:
        if attempts[0]["audit"]["valid"]:
            raise AssertionError("A valid semantic decision must not be retried.")

    execution_record = {
        "case": case_name,
        "model_id": MODEL_IDENTIFIER,
        "upstream_reference": UPSTREAM_REFERENCE,
        "model_load_seconds_this_run": model_load_seconds,
        "inference_seconds_this_run": selected["inference_seconds_this_run"],
        "total_inference_seconds_this_run": generation["total_inference_seconds_this_run"],
        "input_tokens": selected["input_tokens"],
        "generated_tokens": selected["generated_tokens"],
        "input_hashes": {
            preprocessing["before"]["filename"]: preprocessing["before"]["sha256"],
            preprocessing["after"]["filename"]: preprocessing["after"]["sha256"],
            AUDIO_PATH.name: input_hashes[AUDIO_PATH.name],
        },
        "preprocessing": preprocessing,
        "attempts": attempts,
        "retry_used": generation["retry_used"],
        "selected_attempt": generation["selected_attempt"],
        "raw_output": selected["raw_output"],
        "parsed_output": selected["parsed_output"],
        "audit": selected["audit"],
        "human_confirmation_performed": False,
    }
    execution_records[case_name] = execution_record
    record_path = WORK_ROOT / f"execution-{case_name}.json"
    record_path.write_text(json.dumps(execution_record, indent=2, default=str) + "\n", encoding="utf-8")
    print(f"\n=== AUDIT: {case_name} ===")
    print(json.dumps(execution_record, indent=2, default=str))
    print(f"Saved {record_path}")

print("Notebook complete. Each case has a separate execution record. A pending proposal is not a commit; confirmation remains a human application action.")

## Interpreting this run

One pass across the three cases demonstrates the mechanism, not model accuracy. Inspect `execution-bad.json`, `execution-corrected.json`, and `execution-occluded.json` separately. Before publishing a performance statement, run the release protocol on the same approved files and hardware: five consecutive clean-scene proposals and five consecutive bad-scene blocks, plus the occlusion case. Report every failure and the exact environment. Do not copy this notebook's full-checkpoint timing into the browser demo; they are different runtimes.